In [ ]:
# delem=class="floorplan__floorPlanContainer"
# #Format 1
# class="floorplan__floorPlanContainer"
# class="pd__innerHeaderWrapper"
# id="LANDMARK"
# class="review__topWrapper  "
# id="amenitiesWrp"

# #Format 2
# id="FactTableComponent"
# class="pd__innerHeaderWrapper"
# id="LANDMARK"
# class="PremiumPdKeyHighlight__keyHighLights"
# id="AdditionalDetailsComponent"
# id="features"
# class="review__topWrapper  "

In [ ]:
# len(visited_links)

2046

## 100 Pages Scrape

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
import numpy as np
import os
import random
import pyautogui


import pandas as pd
df = pd.read_csv('combined_data.csv')
links_fetched=set(df['link'])

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64)...',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)...',
    'Mozilla/5.0 (X11; Linux x86_64)...'
]

PROXY_USER = '6c57fc4bbe3233db7ed2__cr.np,bd,ru,ae,gb,us,af,in'
PROXY_PASS = '6b028bc0ed2f0ca4'

def create_driver(user_agent):
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--headless=new") #----------------
    options.add_argument(f"user-agent={user_agent}")

    options.add_argument(f"--window-size={random.randint(1200,1600)},{random.randint(800,1000)}")
    service = Service()
    driver = webdriver.Chrome(service=service, options=options)

    driver.execute_cdp_cmd(
        'Page.addScriptToEvaluateOnNewDocument',
        {'source': '''
            Object.defineProperty(navigator, 'webdriver', {
              get: () => undefined
            })
        '''}
    )
    return driver

os.makedirs("data_resales", exist_ok=True)

flat = 3640
skip = 0
visited_links = set()

current_user_agent=random.choice(USER_AGENTS)
agent_session_count = 0
start_time=time.time()


for i in range(95, 96):  # Update as needed
    if i % 3 == 0 or agent_session_count >= 5:
        current_user_agent = random.choice(USER_AGENTS)
        agent_session_count = 0
        print(f"🔄 Changed user-agent to: {current_user_agent}")
    driver = create_driver(current_user_agent)
    wait = WebDriverWait(driver, 10)
# https://www.99acres.com/flats-in-mumbai-ffid
# https://www.99acres.com/property-in-mumbai-without-brokerage-ffid-page-
# https://www.99acres.com/resale-property-in-mumbai-ffid-page-
    try:
        driver.get(f'https://www.99acres.com/property-in-mumbai-without-brokerage-ffid-page-{i}')
        time.sleep(1)
        # driver.minimize_window()
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "tupleNew__outerTupleWrap")))

    except Exception as e:
        print(f"Error on page {i}: {e}")
        driver.quit()
        agent_session_count += 1
        time.sleep(10)
        continue

    containers = driver.find_elements(By.CLASS_NAME, "tupleNew__outerTupleWrap")

    if not containers:
        print(f"⚠ No containers found on page {i}. Trying to go back and reload...")
        driver.back()
        time.sleep(2)
        containers = driver.find_elements(By.CLASS_NAME, "tupleNew__outerTupleWrap")

    print(f"Found {len(containers)} elements on page {i}.")


    # max_sample = min(len(containers), 8)
    # min_sample = min(len(containers), 5)

    # if len(containers) > 0:
    #     sample_size = random.randint(min_sample, max_sample)
    #     containers_idx = random.sample(range(len(containers)), sample_size)
    # else:
    #     containers_idx = []

    containers_idx = list(range(len(containers)))

    page_links = []

    for idx in containers_idx:
        try:
            containers = driver.find_elements(By.CLASS_NAME, "tupleNew__outerTupleWrap")
            elem = containers[idx]

            html_content = elem.get_attribute("outerHTML")
            link_elem = elem.find_element(By.CSS_SELECTOR, "a.tupleNew__propertyHeading.ellipsis")
            link = link_elem.get_attribute("href")
            if link and not link.startswith('http'):
                link = "https://www.99acres.com" + link
            if link and link not in visited_links and link not in links_fetched :
                visited_links.add(link)
                file_path = f"data_resales/flat_{flat}.html"
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(html_content)
                print(f"Container saved: {link}")
                page_links.append((link, file_path))
                flat += 1
            else:
                skip += 1
                print(f"Skipping duplicate link: {skip} --> {link}")

            time.sleep(np.random.uniform(1, 3))

        except Exception as e:
            print(f"Error extracting link at index {idx}: {e}")
            continue

    time.sleep(np.random.uniform(2, 5))

    for link, file_path in page_links:
        try:
            driver.get(link)
            time.sleep(np.random.uniform(1,3))

            wait.until(
                EC.any_of(
                    EC.presence_of_element_located((By.CLASS_NAME, "floorplan__floorPlanContainer")),
                    EC.presence_of_element_located((By.ID, "FactTableComponent"))
                )
            )

            extracted_data = {}
            format_type = "Unknown"
            detail_html = driver.page_source

            if 'class="floorplan__floorPlanContainer"' in detail_html:
                format_type = "Format 1"
                targets = [
                    ("FloorPlan", (By.CLASS_NAME, "floorplan__floorPlanContainer")),
                    ("Header", (By.CLASS_NAME, "pd__innerHeaderWrapper")),
                    ("Landmark", (By.ID, "LANDMARK")),
                    ("Review", (By.CLASS_NAME, "review__topWrapper")),
                    ("Amenities", (By.ID, "amenitiesWrp")),
                    ("FurnishDetails", (By.ID, "FurnishDetails")),
                    ("Features", (By.ID, "features")),
                    ("FeaturesFallback", (By.CLASS_NAME, "component__features pd__pdBlock"))
                ]
            elif 'id="FactTableComponent"' in detail_html:
                format_type = "Format 2"
                targets = [
                    ("FactTable", (By.ID, "FactTableComponent")),
                    ("Header", (By.CLASS_NAME, "pd__innerHeaderWrapper")),
                    ("Landmark", (By.ID, "LANDMARK")),
                    ("Highlights", (By.CLASS_NAME, "PremiumPdKeyHighlight__keyHighLights")),
                    ("AdditionalDetails", (By.ID, "AdditionalDetailsComponent")),
                    ("FurnishDetails", (By.ID, "FurnishDetails")),
                    ("FeaturesSection", (By.XPATH, "//div[@data-label='FACILITIES']")),
                    ("FeaturesFallback", (By.XPATH, "//div[contains(@class, 'component__features') and not(ancestor::div[@id='FurnishDetails'])]")),

                    ("Review", (By.CLASS_NAME, "review__topWrapper"))
                ]
            else:
                targets = []

            for name, locator in targets:
                try:
                    element = driver.find_element(*locator)
                    extracted_data[name] = element.get_attribute("outerHTML")
                except:
                    if name == "Features":
                        try:
                            fallback_elem = driver.find_element(By.CLASS_NAME, "component__features pd__pdBlock")
                            extracted_data["FeaturesFallback"] = fallback_elem.get_attribute("outerHTML")
                            print("✅ Fallback Features extracted.")
                        except:
                            extracted_data["FeaturesFallback"] = "<!-- FeaturesFallback not found -->"
                            print("⚠ FeaturesFallback not found.")
                    else:
                        extracted_data[name] = f"<!-- {name} not found -->"

            with open(file_path, "a", encoding="utf-8") as f:
                f.write(f"\n<!-- Property detail for {link} -->\n")
                f.write(f"<!-- Detected {format_type} -->\n")
                for name, content in extracted_data.items():
                    f.write(f"\n<!-- {name} -->\n{content}\n")

            print(f"✅ Extracted elements for {link} saved.")

        except Exception as e:
            print(f"Error visiting detail page for {link}: {e}")
            time.sleep(10)
    time.sleep(np.random.uniform(1, 3))

    driver.quit()
    agent_session_count += 1
    time.sleep(np.random.uniform(1, 4))

    # if i % 7 == 0:
    #     sleep_time = random.uniform(240, 300)
    #     print(f"🛑 Long break after {i} pages: sleeping for {sleep_time:.1f} sec")
    #     time.sleep(sleep_time)
    # elif i % 3 == 0:
    #     sleep_time = random.uniform(20,45 )
    #     print(f"🛑 Long break after {i} pages: sleeping for {sleep_time:.1f} sec")
    #     time.sleep(sleep_time)
    

end_time=time.time() 
print(start_time-end_time)

## Next Page Button Scrape

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
import numpy as np
import os
import random
import pandas as pd

# Load existing data to skip already scraped links
df = pd.read_csv('combined_data.csv')
links_fetched = set(df['link'])

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64)...',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)...',
    'Mozilla/5.0 (X11; Linux x86_64)...'
]

def create_driver(user_agent):
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(f"user-agent={user_agent}")
    options.add_argument(f"--window-size={random.randint(1200,1600)},{random.randint(800,1000)}")
    service = Service()
    driver = webdriver.Chrome(service=service, options=options)
    driver.execute_cdp_cmd(
        'Page.addScriptToEvaluateOnNewDocument',
        {'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'}
    )
    return driver

def click_next_page(driver, wait, max_scrolls=25):
    for scroll_count in range(max_scrolls):
        try:
            next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//a[contains(text(), "Next Page")]')))
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
            time.sleep(1)
            try:
                next_button.click()
                print("✅ Clicked 'Next Page' using normal click.")
            except:
                driver.execute_script("arguments[0].click();", next_button)
                print("✅ Clicked 'Next Page' using JavaScript click.")
            return True
        except:
            driver.execute_script("window.scrollBy(0, 3000);")
            print(f"🔃 Scrolling... (attempt {scroll_count + 1})")
            time.sleep(1)
    print("❌ Failed to find/click 'Next Page'.")
    return False

# Setup
os.makedirs("data_resales", exist_ok=True)
visited_links = set()
flat = 6308
skip = 0
i = 100

current_user_agent = random.choice(USER_AGENTS)
driver = create_driver(current_user_agent)
wait = WebDriverWait(driver, 10)
driver.get(f'https://www.99acres.com/flats-in-mumbai-ffid-page-{i}')
start_time = time.time()

while True:
    try:
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "tupleNew__outerTupleWrap")))
    except Exception as e:
        print(f"⚠️ Error on page {i}: {e}")
        continue

    containers = driver.find_elements(By.CLASS_NAME, "tupleNew__outerTupleWrap")
    if not containers:
        print(f"⚠ No containers found on page {i}. Trying to reload...")
        time.sleep(2)
        continue

    print(f"📦 Found {len(containers)} containers on page {i}")
    for idx, elem in enumerate(containers):
        try:
            html_content = elem.get_attribute("outerHTML")
            link_elem = elem.find_element(By.CSS_SELECTOR, "a.tupleNew__propertyHeading.ellipsis")
            link = link_elem.get_attribute("href")
            if link and not link.startswith('http'):
                link = "https://www.99acres.com" + link

            if link and link not in visited_links and link not in links_fetched:
                visited_links.add(link)
                file_path = f"data_flats/flat_{flat}.html"
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(html_content)
                print(f"✅ Container saved: {link}")
                flat += 1
            else:
                skip += 1
                print(f"⏩ Skipping duplicate ({skip}): {link}")

            time.sleep(np.random.uniform(1, 3))
        except Exception as e:
            print(f"❌ Error extracting container index {idx}: {e}")
            continue

    # Move to next page
    if not click_next_page(driver, wait):
        break

    i += 1
    if i % 3 == 0:
        sleep_time = random.uniform(10, 15)
        print(f"🛑 Short break after {i} pages: sleeping for {sleep_time:.1f} sec")
        time.sleep(sleep_time)

driver.quit()
end_time = time.time()
print(f"⏱ Total scrape time: {(end_time - start_time)/60:.2f} minutes")


📦 Found 25 containers on page 100
✅ Container saved: https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-beverly-hills-andheri-west-mumbai-andheri-dahisar-1300-sq-ft-spid-Y84452732
✅ Container saved: https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-rna-ng-eclat-andheri-west-mumbai-andheri-dahisar-863-sq-ft-spid-G84338248
✅ Container saved: https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-dlh-leo-tower-andheri-west-mumbai-andheri-dahisar-640-sq-ft-spid-U84286722
✅ Container saved: https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-dlh-leo-tower-andheri-west-mumbai-andheri-dahisar-660-sq-ft-spid-M84284816
✅ Container saved: https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-dlh-leo-tower-andheri-west-mumbai-andheri-dahisar-670-sq-ft-spid-V84284674
✅ Container saved: https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-highland-park-mumbai-andheri-dahisar-1300-sq-ft-spid-C84271588
✅ Container saved: https

KeyboardInterrupt: 

#### Scrape Append Data to HTML files 

In [1]:
import os
import time
import random
import numpy as np
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# --- Setup ---
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64)...',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)...',
    'Mozilla/5.0 (X11; Linux x86_64)...'
]


def create_driver(user_agent):
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(f"user-agent={user_agent}")
    options.add_argument("--window-size=1280,1000")
    service = Service()
    driver = webdriver.Chrome(service=service, options=options)
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver


def extract_link_from_html(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
    a_tag = soup.find("a", class_="tupleNew__propertyHeading")
    return a_tag['href'] if a_tag else None


def extract_details_and_append(driver, wait, link, file_path):
    try:
        driver.get(link)
        time.sleep(np.random.uniform(1, 3))

        wait.until(
            EC.any_of(
                EC.presence_of_element_located((By.CLASS_NAME, "floorplan__floorPlanContainer")),
                EC.presence_of_element_located((By.ID, "FactTableComponent"))
            )
        )

        extracted_data = {}
        format_type = "Unknown"
        detail_html = driver.page_source

        if 'class="floorplan__floorPlanContainer"' in detail_html:
            format_type = "Format 1"
            targets = [
                ("FloorPlan", (By.CLASS_NAME, "floorplan__floorPlanContainer")),
                ("Header", (By.CLASS_NAME, "pd__innerHeaderWrapper")),
                ("Landmark", (By.ID, "LANDMARK")),
                ("Review", (By.CLASS_NAME, "review__topWrapper")),
                ("Amenities", (By.ID, "amenitiesWrp")),
                ("FurnishDetails", (By.ID, "FurnishDetails")),
                ("Features", (By.ID, "features")),
                ("FeaturesFallback", (By.CLASS_NAME, "component__features pd__pdBlock"))
            ]
        elif 'id="FactTableComponent"' in detail_html:
            format_type = "Format 2"
            targets = [
                ("FactTable", (By.ID, "FactTableComponent")),
                ("Header", (By.CLASS_NAME, "pd__innerHeaderWrapper")),
                ("Landmark", (By.ID, "LANDMARK")),
                ("Highlights", (By.CLASS_NAME, "PremiumPdKeyHighlight__keyHighLights")),
                ("AdditionalDetails", (By.ID, "AdditionalDetailsComponent")),
                ("FurnishDetails", (By.ID, "FurnishDetails")),
                ("FeaturesSection", (By.XPATH, "//div[@data-label='FACILITIES']")),
                ("FeaturesFallback", (By.XPATH, "//div[contains(@class, 'component__features') and not(ancestor::div[@id='FurnishDetails'])]")),
                ("Review", (By.CLASS_NAME, "review__topWrapper"))
            ]
        else:
            targets = []

        for name, locator in targets:
            try:
                element = driver.find_element(*locator)
                extracted_data[name] = element.get_attribute("outerHTML")
            except:
                if name == "Features":
                    try:
                        fallback_elem = driver.find_element(By.CLASS_NAME, "component__features pd__pdBlock")
                        extracted_data["FeaturesFallback"] = fallback_elem.get_attribute("outerHTML")
                        print("✅ Fallback Features extracted.")
                    except:
                        extracted_data["FeaturesFallback"] = "<!-- FeaturesFallback not found -->"
                        print("⚠ FeaturesFallback not found.")
                else:
                    extracted_data[name] = f"<!-- {name} not found -->"

        with open(file_path, "a", encoding="utf-8") as f:
            f.write(f"\n<!-- Property detail for {link} -->\n")
            f.write(f"<!-- Detected {format_type} -->\n")
            for name, content in extracted_data.items():
                f.write(f"\n<!-- {name} -->\n{content}\n")

        print(f"✅ Extracted elements for {link} saved.")

    except Exception as e:
        print(f"❌ Error visiting detail page for {link}: {e}")
        time.sleep(10)


# === Main Driver ===
directory = "data_flats"
all_files = sorted([f for f in os.listdir(directory) if f.endswith('.html')])

user_agent = random.choice(USER_AGENTS)
driver = create_driver(user_agent)
wait = WebDriverWait(driver, 10)

for file_name in all_files:
    file_path = os.path.join(directory, file_name)
    print(f"\n📂 Processing: {file_name}")
    link = extract_link_from_html(file_path)

    if link:
        if not link.startswith("http"):
            link = "https://www.99acres.com" + link
        extract_details_and_append(driver, wait, link, file_path)
        time.sleep(random.uniform(2, 4))
    else:
        print(f"⚠️ No link found in {file_name}")

driver.quit()


📂 Processing: flat_7280.html
✅ Extracted elements for https://www.99acres.com/1-bhk-bedroom-apartment-flat-for-sale-in-kipl-morya-kasar-vadavali-mumbai-thane-723-sq-ft-r6-spid-X73430159 saved.

📂 Processing: flat_7281.html
✅ Extracted elements for https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-lodha-splendora-ghodbunder-road-mumbai-thane-771-sq-ft-spid-I83873038 saved.

📂 Processing: flat_7282.html
✅ Extracted elements for https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-powai-central-mumbai-suburbs-689-sq-ft-r3-spid-K77547827 saved.

📂 Processing: flat_7283.html
✅ Extracted elements for https://www.99acres.com/1-bhk-bedroom-apartment-flat-for-sale-in-unique-ivana-mira-road-mira-bhayandar-390-sq-ft-spid-O83859564 saved.

📂 Processing: flat_7284.html
✅ Extracted elements for https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-hiranandani-lake-enclave-hiranandani-estate-mumbai-thane-1194-sq-ft-spid-H83877468 saved.

📂 Processing: flat_7285.